# Data Cleaning & Transformation Pipeline
This notebook performs progressive cleaning on `Airbnb_Open_Data.csv`. 
Every transformation is documented in memory via a structured log and verified immediately.


In [19]:
from pathlib import Path
import numpy as np
import pandas as pd

# Global audit holder
LOG = []


def log(
    step: str,
    problem: str,
    column: str,
    rows_affected,
    what_i_did: str,
    method: str,
) -> None:
  LOG.append({
      "step": step,
      "problem": problem,
      "column": column,
      "rows_affected": rows_affected,
      "what_i_did": what_i_did,
      "method": method,
  })
  print(f"[{step}] {problem}: {rows_affected} -> {what_i_did}")


# Load the raw dataset
df = pd.read_csv("data/Airbnb_Open_Data.csv", low_memory=False)
initial_shape = df.shape
print(f"Initial Dataset Shape: {initial_shape}")

Initial Dataset Shape: (102599, 26)


### Reading the dataset

In [16]:
messy = pd.read_csv('data/Airbnb_Open_Data.csv')


C:\Users\user\AppData\Local\Temp\ipykernel_11016\32585487.py:1: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  messy = pd.read_csv('data/Airbnb_Open_Data.csv')


### Checking the data's profile

In [11]:
profile = ProfileReport(messy, title='Airbnb Open Data Profiling Report', explorative=True)
profile.to_file('report.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 26/26 [00:22<00:00,  1.15it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

### View the profile report

In [ ]:
profile.to_notebook_iframe()

## Key findings from the report

1. Number of variables (Columns) is 26
2. Number of observations (Rows) is 102599
3. Missing cells is 190769 which is about 7.2% of the dataset
4. Duplicate rows is 541 which is about 0.5% of all rows
5. We have 10 Numeric Columns, 7 text and 7 category columns, 1 boolean column and 1 datetime column

## Data Quality Issues

1. Inconsistent Column names
2. Lincense column is empyty, so i will drop it
3. Construction year is stored as 
4. Availability has 3677 days and -10 days, since days shouldn't be less than 0 and it's kind of impossible to be more than a ear
5. Last Review has 2058, that's impossible
6. I have minimum nights that are less than 0 and some that are above a year
7. manhatan and brookin for Manhattan and Brooklyn, need to be changed

## Phase 2: Structural Integrity
- Standardize messy column headers into clean `snake_case`.
- Drop exact row duplicates and deduplicate conflicting `id` values.
- Prune zero-variance and sparsely populated columns.

In [20]:
# --- 2a. Tidy column names ---
def to_snake_case(name: str) -> str:
  return name.strip().lower().replace(" ", "_")


old_names = list(df.columns)
df.columns = [to_snake_case(c) for c in old_names]
renamed = sum(1 for o, n in zip(old_names, df.columns) if o != n)
log(
    "2a",
    "Messy column names",
    "all columns",
    renamed,
    "Standardized to snake_case",
    "str.strip().str.lower().str.replace()",
)

# --- 2b. Remove duplicate listings ---
before = len(df)
df = df.drop_duplicates(keep="first")
removed = before - len(df)
log(
    "2b",
    "Duplicate listings (identical rows)",
    "all columns",
    removed,
    f"Dropped duplicates ({before:,} -> {len(df):,})",
    "drop_duplicates(keep='first')",
)

# Secondary ID-level safety check
still_repeated = int(df["id"].duplicated().sum())
if still_repeated > 0:
  df = df.drop_duplicates(subset=["id"], keep="first")
  log(
      "2b-2",
      "Conflicting duplicate IDs",
      "id",
      still_repeated,
      "Retained first listing per ID",
      "drop_duplicates(subset=['id'], keep='first')",
  )

# --- 2c. Drop non-informative columns ---
drops = {
    "license": "Over 99.9% missing cells",
    "country": "Constant value ('United States')",
    "country_code": "Constant value ('US')",
    "house_rules": "Sparse unstructured text (>50% nulls)",
}
for col, reason in drops.items():
  if col in df.columns:
    filled = int(df[col].notna().sum())
    log(
        "2c",
        f"Useless column ({reason})",
        col,
        f"{filled:,} filled cells",
        "Dropped column",
        "df.drop(columns=...)",
    )

df = df.drop(columns=[c for c in drops if c in df.columns])

# Verification output
df.info()

[2a] Messy column names: 15 -> Standardized to snake_case
[2b] Duplicate listings (identical rows): 541 -> Dropped duplicates (102,599 -> 102,058)
[2c] Useless column (Over 99.9% missing cells): 2 filled cells -> Dropped column
[2c] Useless column (Constant value ('United States')): 101,526 filled cells -> Dropped column
[2c] Useless column (Constant value ('US')): 101,927 filled cells -> Dropped column
[2c] Useless column (Sparse unstructured text (>50% nulls)): 50,216 filled cells -> Dropped column
<class 'pandas.core.frame.DataFrame'>
Index: 102058 entries, 0 to 102057
Data columns (total 22 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102058 non-null  int64  
 1   name                            101808 non-null  object 
 2   host_id                         102058 non-null  int64  
 3   host_identity_verified          101769 non-null  object 
 4   host_name  

## Phase 3: Type Parsing & Format Harmonization
- Strip currency formatting (`$`, commas, spaces) from financial fields and cast to float.
- Parse text date entries in `last_review` into pandas `datetime64[ns]` timestamps.

In [21]:
# --- 3a. Clean currency fields ---
for col in ["price", "service_fee"]:
  if col in df.columns:
    mask_dirty = df[col].astype(str).str.contains(r"[\$, ]", regex=True)
    affected = int(mask_dirty.sum())
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r"[\$, ]", "", regex=True)
        .replace({"nan": np.nan, "": np.nan})
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")
    log(
        "3a",
        "Currency formatting in numeric field",
        col,
        affected,
        "Stripped symbols/whitespace and converted to float",
        "str.replace() + pd.to_numeric()",
    )

# --- 3b. Convert last_review to datetime ---
if "last_review" in df.columns:
  before_nans = int(df["last_review"].isna().sum())
  df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")
  new_nans = int(df["last_review"].isna().sum()) - before_nans
  log(
      "3b",
      "String representation of dates",
      "last_review",
      f"{len(df) - before_nans:,} strings",
      f"Converted to datetime (unparseable coerced to NaT: {new_nans})",
      "pd.to_datetime(errors='coerce')",
  )

# Verification output
df[["price", "service_fee", "last_review"]].describe(include="all")

[3a] Currency formatting in numeric field: 101811 -> Stripped symbols/whitespace and converted to float
[3a] Currency formatting in numeric field: 101785 -> Stripped symbols/whitespace and converted to float
[3b] String representation of dates: 86,226 strings -> Converted to datetime (unparseable coerced to NaT: 0)


,price,service_fee,last_review
count,101811.000000,101785.000000,86226
mean,625.355580,125.039249,2019-06-11 02:12:22.996312064
min,50.000000,10.000000,2012-07-11 00:00:00
25%,340.000000,68.000000,2018-10-27 00:00:00
50%,625.000000,125.000000,2019-06-13 00:00:00
75%,913.000000,183.000000,2019-07-05 00:00:00
max,1200.000000,240.000000,2058-06-16 00:00:00
std,331.672649,66.325905,NaN


## Phase 4: Categorical Consistency
- Clean borough misspellings (`Brookln`, `Manhatan`) and standardize to Title Case.
- Strip excess whitespace and normalize casing across categorical metadata.

In [22]:
# --- 4a. Fix borough names ---
if "neighbourhood_group" in df.columns:
  df["neighbourhood_group"] = (
      df["neighbourhood_group"].astype(str).str.strip().str.title()
  )
  typo_map = {"Brookln": "Brooklyn", "Manhatan": "Manhattan"}
  typos_count = int(df["neighbourhood_group"].isin(typo_map.keys()).sum())

  df["neighbourhood_group"] = df["neighbourhood_group"].replace(typo_map)
  df["neighbourhood_group"] = df["neighbourhood_group"].replace(
      {"Nan": np.nan, "None": np.nan}
  )
  log(
      "4a",
      "Borough misspellings ('Brookln', 'Manhatan')",
      "neighbourhood_group",
      typos_count,
      "Fixed spelling errors and unified title casing",
      "replace(typo_map)",
  )

# --- 4b. Standardize text categories ---
categories = [
    "host_identity_verified",
    "cancellation_policy",
    "room_type",
    "instant_bookable",
]
for col in categories:
  if col in df.columns:
    df[col] = df[col].astype(str).str.strip().str.lower()
    df[col] = df[col].replace({"nan": np.nan, "none": np.nan, "": np.nan})
    log(
        "4b",
        "Inconsistent text casing / pseudo-nulls",
        col,
        "all rows checked",
        "Lowercased string values and converted 'nan'/'none' to np.nan",
        "str.strip().str.lower().replace()",
    )

# Verification output
print("Neighbourhood Groups:", df["neighbourhood_group"].unique())
print("Room Types:", df["room_type"].unique())

[4a] Borough misspellings ('Brookln', 'Manhatan'): 2 -> Fixed spelling errors and unified title casing
[4b] Inconsistent text casing / pseudo-nulls: all rows checked -> Lowercased string values and converted 'nan'/'none' to np.nan
[4b] Inconsistent text casing / pseudo-nulls: all rows checked -> Lowercased string values and converted 'nan'/'none' to np.nan
[4b] Inconsistent text casing / pseudo-nulls: all rows checked -> Lowercased string values and converted 'nan'/'none' to np.nan
[4b] Inconsistent text casing / pseudo-nulls: all rows checked -> Lowercased string values and converted 'nan'/'none' to np.nan
Neighbourhood Groups: ['Brooklyn' 'Manhattan' 'Queens' nan 'Staten Island' 'Bronx']
Room Types: ['private room' 'entire home/apt' 'shared room' 'hotel room']


## Phase 5: Domain Constraints & Temporal Sanity
- Bound `availability_365` within the realistic calendar window: $[0, 365]$.
- Filter negative and excessive values from `minimum_nights`.
- Filter out-of-range building construction years and review timestamps set in the future.

In [ ]:
# --- 5a. Availability bounds (0 <= days <= 365) ---
if "availability_365" in df.columns:
  invalid_avail = (df["availability_365"] < 0) | (df["availability_365"] > 365)
  affected_avail = int(invalid_avail.sum())
  df.loc[invalid_avail, "availability_365"] = np.nan
  log(
      "5a",
      "Availability outside calendar year (<0 or >365)",
      "availability_365",
      affected_avail,
      "Set out-of-bounds days to NaN",
      "df.loc[mask] = np.nan",
  )

# --- 5b. Minimum nights validity ---
if "minimum_nights" in df.columns:
  invalid_nights = (df["minimum_nights"] <= 0) | (df["minimum_nights"] > 365)
  affected_nights = int(invalid_nights.sum())
  df.loc[invalid_nights, "minimum_nights"] = np.nan
  log(
      "5b",
      "Negative or extreme (>365) minimum nights",
      "minimum_nights",
      affected_nights,
      "Set invalid minimum night requirements to NaN",
      "df.loc[mask] = np.nan",
  )

# --- 5c. Historical building construction year ---
if "construction_year" in df.columns:
  df["construction_year"] = pd.to_numeric(
      df["construction_year"], errors="coerce"
  )
  current_year = 2026
  invalid_years = (df["construction_year"] < 1900) | (
      df["construction_year"] > current_year
  )
  affected_years = int(invalid_years.sum())
  df.loc[invalid_years, "construction_year"] = np.nan
  log(
      "5c",
      f"Construction year out of reasonable scope (<1900 or >{current_year})",
      "construction_year",
      affected_years,
      "Coerced invalid years to NaN",
      "df.loc[mask] = np.nan",
  )

# --- 5d. Future dates in last_review ---
if "last_review" in df.columns:
  SNAPSHOT_DATE = pd.Timestamp("2022-08-01")   # the file was published in Aug 2022, so a later review is impossible
  future_dates = df["last_review"] > SNAPSHOT_DATE
  affected_dates = int(future_dates.sum())
  df.loc[future_dates, "last_review"] = pd.NaT
  log(
      "5d",
      "Reviews stamped with impossible future dates (e.g. 2058)",
      "last_review",
      affected_dates,
      "Replaced future review timestamps with NaT",
      "df.loc[mask] = pd.NaT",
  )

# Verification output
df[["availability_365", "minimum_nights", "construction_year"]].describe()

[5a] Availability outside calendar year (<0 or >365): 3185 -> Set out-of-bounds days to NaN
[5b] Negative or extreme (>365) minimum nights: 48 -> Set invalid minimum night requirements to NaN
[5c] Construction year out of reasonable scope (<1900 or >2026): 0 -> Coerced invalid years to NaN
[5d] Reviews stamped with impossible future dates (e.g. 2058): 2 -> Replaced future review timestamps with NaT


,availability_365,minimum_nights,construction_year
count,98425.000000,101610.000000,101844.000000
mean,134.528484,7.845842,2012.488090
std,129.805264,17.065232,5.765838
min,0.000000,1.000000,2003.000000
25%,2.000000,2.000000,2007.000000
50%,90.000000,3.000000,2012.000000
75%,254.000000,5.000000,2017.000000
max,365.000000,365.000000,2022.000000


## Phase 6: Logical Imputation & Deliverable Export
- Impute `reviews_per_month` as `0.0` for listings with `number_of_reviews == 0`.
- Reset DataFrame indices.
- Persist `airbnb_clean.csv` and export `cleaning_log.csv`.

In [25]:
# --- 6a. Logical review imputation ---
if "reviews_per_month" in df.columns and "number_of_reviews" in df.columns:
  zero_review_mask = (df["number_of_reviews"] == 0) & (
      df["reviews_per_month"].isna()
  )
  affected_reviews = int(zero_review_mask.sum())
  df.loc[zero_review_mask, "reviews_per_month"] = 0.0
  log(
      "6a",
      "Missing monthly rate on listings with zero total reviews",
      "reviews_per_month",
      affected_reviews,
      "Imputed 0.0 instead of NaN",
      "df.loc[mask, 'reviews_per_month'] = 0.0",
  )

# Reset row index
df = df.reset_index(drop=True)

# Save deliverables
out = Path("output")
out.mkdir(parents=True, exist_ok=True)

df.to_csv(out / "airbnb_clean.csv", index=False)
log_df = pd.DataFrame(LOG)
log_df.to_csv(out / "cleaning_log.csv", index=False)

print("\n--- Pipeline Summary ---")
print(f"Initial Dimensions: {initial_shape}")
print(f"Final Dimensions:   {df.shape}")
print(f"Total Logged Steps: {len(log_df)}")

[6a] Missing monthly rate on listings with zero total reviews: 15673 -> Imputed 0.0 instead of NaN

--- Pipeline Summary ---
Initial Dimensions: (102599, 26)
Final Dimensions:   (102058, 22)
Total Logged Steps: 19


In [26]:
# Display the completed audit ledger directly in the notebook
log_df

,step,problem,column,rows_affected,what_i_did,method
0,2a,Messy column names,all columns,15,Standardized to snake_case,str.strip().str.lower().str.replace()
1,2b,Duplicate listings (identical rows),all columns,541,"Dropped duplicates (102,599 -> 102,058)",drop_duplicates(keep='first')
2,2c,Useless column (Over 99.9% missing cells),license,2 filled cells,Dropped column,df.drop(columns=...)
3,2c,Useless column (Constant value ('United States')),country,"101,526 filled cells",Dropped column,df.drop(columns=...)
4,2c,Useless column (Constant value ('US')),country_code,"101,927 filled cells",Dropped column,df.drop(columns=...)
5,2c,Useless column (Sparse unstructured text (>50%...,house_rules,"50,216 filled cells",Dropped column,df.drop(columns=...)
6,3a,Currency formatting in numeric field,price,101811,Stripped symbols/whitespace and converted to f...,str.replace() + pd.to_numeric()
7,3a,Currency formatting in numeric field,service_fee,101785,Stripped symbols/whitespace and converted to f...,str.replace() + pd.to_numeric()
8,3b,String representation of dates,last_review,"86,226 strings",Converted to datetime (unparseable coerced to ...,pd.to_datetime(errors='coerce')
9,4a,"Borough misspellings ('Brookln', 'Manhatan')",neighbourhood_group,2,Fixed spelling errors and unified title casing,replace(typo_map)
